### Import Packages

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

from transformers import BertTokenizerFast
from datasets import load_dataset

import numpy as np
import matplotlib.pyplot as plt
from torchmetrics import Accuracy

from tqdm import tqdm

### 1.1 Load Dataset

In [8]:
from datasets import load_dataset

In [9]:
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

In [10]:
dataset.save_to_disk(r"../RNN_NLP_Project/datasets")

Saving the dataset (1/1 shards): 100%|██████████| 2000/2000 [00:00<00:00, 154213.69 examples/s]


In [12]:
from datasets import load_from_disk

dataset = load_from_disk(
    r"../RNN_NLP_Project/datasets"
)

In [13]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

### 1.2 Inspect the Dataset 

In [21]:
label_names = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

In [22]:
for i in range(5):
    example = dataset["train"][i]
    print(f"Example {i + 1}")
    print("Text:", example["text"])
    print("Label:", example["label"])
    print("Sentiment:", label_names[example["label"]])
    print("-" * 50)

Example 1
Text: "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"
Label: 2
Sentiment: Positive
--------------------------------------------------
Example 2
Text: "Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"
Label: 1
Sentiment: Neutral
--------------------------------------------------
Example 3
Text: Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.
Label: 1
Sentiment: Neutral
--------------------------------------------------
Example 4
Text: Chase Headley's RBI double in the 8th inning off David Price snapped a Yankees streak of 33 consecutive scoreless innings against Blue Jays
Label: 1
Sentiment: Neutral
--------------------------------------------------
Example 5
Text: @user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017"
Label: 2
Sentiment

### 1.3 Preprocessing

In [14]:
train_dataset = dataset["train"]
valid_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [15]:
train_dataset.shape

(45615, 2)

In [16]:
valid_dataset.shape

(2000, 2)

In [17]:
test_dataset.shape

(12284, 2)

In [18]:
print(train_dataset.column_names)
print(valid_dataset.column_names)
print(test_dataset.column_names)


['text', 'label']
['text', 'label']
['text', 'label']


In [19]:
from transformers import BertTokenizerFast

In [20]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [ ]:
tokenizer(["NLP Project Finish Soon"])

{'input_ids': [[101, 17953, 2361, 2622, 3926, 2574, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1]]}

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128 
    )

### Mapping

In [ ]:
train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 12284/12284 [00:01<00:00, 7102.12 examples/s]


In [ ]:
train_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 45615
})

In [ ]:
valid_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})

In [ ]:
test_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 12284
})

In [ ]:
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
valid_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

### DataLoader

In [ ]:
BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
x = next(iter(train_loader))

In [ ]:
batch = next(iter(valid_loader))
batch

{'label': tensor([1, 2, 0, 1]),
 'input_ids': tensor([[  101,  2601,  9293,  1017,  2258,  4888,  3058,  4484,  2007,  2047,
           9117,  1024,  9979,  1996,  4768,  1012,   102,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,   

In [ ]:
print(batch["input_ids"].shape)

torch.Size([4, 128])


In [ ]:
input_ids = x['input_ids'].long()
input_ids

tensor([[  101,  1045,  2215,  2000,  3637,  2021,  4422,  2050,  7459, 22868,
          2066,  2045,  1005,  1055,  2053, 13847,  4826,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  